In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

In [ ]:
MODEL_PATH = Path("../models/isolation_forest.pkl")
PREDICTION_PATH = Path("../outputs/anomaly_predictions.csv")

GROUND_TRUTH_PATH = Path("../../data/processed/audit_ground.csv")

In [ ]:
predictions = pd.read_csv(PREDICTION_PATH)

print("Prediction shape:", predictions.shape)
predictions.head()

In [ ]:
ground_truth = pd.read_csv(GROUND_TRUTH_PATH)

print("Ground truth shape:", ground_truth.shape)
ground_truth.head()

In [ ]:
print("Prediction columns:")
print(predictions.columns.tolist())

print("\nGround truth columns:")
print(ground_truth.columns.tolist())

In [ ]:
evaluation = predictions.merge(
    ground_truth,
    on="invoice_id",
    how="inner"
)

print("Evaluation shape:", evaluation.shape)

In [ ]:
print("Total predictions:", len(predictions))
print("Total ground truth:", len(ground_truth))
print("Matched records:", len(evaluation))

In [ ]:
y_true = evaluation["has_injected_error"].astype(int)

y_pred = evaluation["is_anomaly"].astype(int)

print("Actual labels:")
print(y_true.value_counts())

print("\nPredicted labels:")
print(y_pred.value_counts())

In [ ]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

print(f"Accuracy: {accuracy:.4f}")

In [ ]:
precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

print(f"Precision: {precision:.4f}")

In [ ]:
recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

print(f"Recall: {recall:.4f}")

In [ ]:
f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

print(f"F1-score: {f1:.4f}")

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Normal", "Error"],
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

print(cm)

In [ ]:
cm_df = pd.DataFrame(
    cm,
    index=["Actual Normal", "Actual Error"],
    columns=["Predicted Normal", "Predicted Error"]
)

cm_df

In [ ]:
actual_errors = evaluation[
    evaluation["has_injected_error"] == 1
].copy()

print("Actual errors:", len(actual_errors))

In [ ]:
detected_errors = evaluation[
    (evaluation["has_injected_error"] == 1) &
    (evaluation["is_anomaly"] == 1)
].copy()

print("Detected errors:", len(detected_errors))

In [ ]:
missed_errors = evaluation[
    (evaluation["has_injected_error"] == 1) &
    (evaluation["is_anomaly"] == 0)
].copy()

print("Missed errors:", len(missed_errors))

In [ ]:
false_positives = evaluation[
    (evaluation["has_injected_error"] == 0) &
    (evaluation["is_anomaly"] == 1)
].copy()

print("False positives:", len(false_positives))

In [ ]:
error_cases = evaluation[
    evaluation["has_injected_error"] == 1
].copy()
error_type_performance = (
    error_cases
    .groupby("injected_error_type")
    .agg(
        total_errors=("invoice_id", "count"),
        detected_errors=("is_anomaly", "sum")
    )
)
error_type_performance["detection_rate"] = (
    error_type_performance["detected_errors"] /
    error_type_performance["total_errors"]
)